### Torch CUDA and Imports

In [1]:
!pip install transformers==4.44.0 datasets==3.6.0 scikit-learn accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 68.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.5 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have n

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
Tesla T4


### SciFAct Leakage Identification and Fix

In [4]:
import json
from pathlib import Path
from collections import Counter

SCIFACT_DIR = Path("/kaggle/input/datasets/minoola33/scifact-data")
LABEL2ID = {"SUPPORT": 0, "NOT_ENOUGH_INFO": 1, "CONTRADICT": 2}

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def build_corpus_lookup(corpus_path):
    lookup = {}
    for row in load_jsonl(corpus_path):
        lookup[int(row["doc_id"])] = row["abstract"]
    return lookup

def flatten_claims(claims, corpus):
    premises, hypotheses, labels_out, doc_ids_used = [], [], [], []
    for claim in claims:
        claim_text = claim["claim"]
        evidence   = claim.get("evidence", {})
        if not evidence:
            premises.append(claim_text)
            hypotheses.append("")
            labels_out.append(LABEL2ID["NOT_ENOUGH_INFO"])
            doc_ids_used.append(set())
            continue
        for doc_id_str, entries in evidence.items():
            doc_id   = int(doc_id_str)
            abstract = corpus.get(doc_id, [])
            for entry in entries:
                sent_idx      = entry["sentences"][0] if entry["sentences"] else 0
                evidence_text = abstract[sent_idx] if sent_idx < len(abstract) else ""
                premises.append(claim_text)
                hypotheses.append(evidence_text)
                labels_out.append(LABEL2ID[entry["label"]])
                doc_ids_used.append({doc_id})
    return {"premise": premises, "hypothesis": hypotheses, "labels": labels_out}, doc_ids_used

corpus       = build_corpus_lookup(SCIFACT_DIR / "corpus.jsonl")
claims_train = load_jsonl(SCIFACT_DIR / "claims_train.jsonl")
claims_dev   = load_jsonl(SCIFACT_DIR / "claims_dev.jsonl")

train_flat, train_doc_ids = flatten_claims(claims_train, corpus)
dev_flat,   dev_doc_ids   = flatten_claims(claims_dev,   corpus)

train_hyp_set = set(h for h in train_flat["hypothesis"] if h)
dev_hyp_set   = set(h for h in dev_flat["hypothesis"]   if h)
overlapping_sentences = train_hyp_set & dev_hyp_set
dev_rows_affected     = sum(1 for h in dev_flat["hypothesis"] if h and h in train_hyp_set)

print(f"=== Sentence-level overlap ===")
print(f"  Unique train hypothesis sentences : {len(train_hyp_set)}")
print(f"  Unique dev   hypothesis sentences : {len(dev_hyp_set)}")
print(f"  Overlapping sentences             : {len(overlapping_sentences)}")
print(f"  Dev rows that contain a leaked sentence: {dev_rows_affected} / {len(dev_flat['hypothesis'])}")

train_docs  = set(did for s in train_doc_ids for did in s)
dev_docs    = set(did for s in dev_doc_ids   for did in s)
shared_docs = train_docs & dev_docs

print(f"\n=== Doc-id level overlap ===")
print(f"  Unique doc_ids in train : {len(train_docs)}")
print(f"  Unique doc_ids in dev   : {len(dev_docs)}")
print(f"  Shared doc_ids          : {len(shared_docs)}")

print(f"\n=== Label distribution ===")
for split_name, flat in [("train", train_flat), ("dev", dev_flat)]:
    c = Counter(flat["labels"])
    print(f"  {split_name}: SUPPORT={c[0]}  NEI={c[1]}  CONTRADICT={c[2]}  total={len(flat['labels'])}")

dev_hyp_list = dev_flat["hypothesis"]
still_leaking = sum(1 for h in dev_hyp_list if h and h in train_hyp_set)
print(f"\nTrain sentences still appearing in dev: {still_leaking}")
clean_dev_indices = [
    i for i, h in enumerate(dev_flat["hypothesis"])
    if not h or h not in train_hyp_set
]
print(f"Clean dev rows: {len(clean_dev_indices)} / {len(dev_flat['hypothesis'])}")

=== Sentence-level overlap ===
  Unique train hypothesis sentences : 683
  Unique dev   hypothesis sentences : 300
  Overlapping sentences             : 158
  Dev rows that contain a leaked sentence: 173 / 450

=== Doc-id level overlap ===
  Unique doc_ids in train : 365
  Unique doc_ids in dev   : 182
  Shared doc_ids          : 110

=== Label distribution ===
  train: SUPPORT=616  NEI=304  CONTRADICT=341  total=1261
  dev: SUPPORT=216  NEI=112  CONTRADICT=122  total=450

Train sentences still appearing in dev: 173
Clean dev rows: 277 / 450


### Fine-Tuning on SciFAct

In [5]:
# ── torch.load patch (suppress weights_only warning on older torch) ───────────
import torch

_orig_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# ── Standard imports ──────────────────────────────────────────────────────────
import json
import numpy as np
from pathlib import Path
from collections import Counter
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, classification_report
from torch.nn import CrossEntropyLoss

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# ── Paths ─────────────────────────────────────────────────────────────────────
MULTINLI_CKPT = "/kaggle/input/models/minoola33/deberta-v3-base-multinli-nli/transformers/default/1"
OUTPUT_DIR    = "/kaggle/working/scifact_checkpoint"

# Folder that contains claims_train.jsonl / claims_dev.jsonl / corpus.jsonl
SCIFACT_DIR   = Path("/kaggle/input/datasets/minoola33/scifact-data")

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LENGTH   = 256
BATCH_SIZE   = 8
EPOCHS       = 8
LR           = 2e-6
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

# ── Label mapping ─────────────────────────────────────────────────────────────
LABEL2ID = {"SUPPORT": 0, "NOT_ENOUGH_INFO": 1, "CONTRADICT": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_jsonl(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def build_corpus_lookup(corpus_path: Path) -> dict[int, list[str]]:
    lookup = {}
    for row in load_jsonl(corpus_path):
        lookup[int(row["doc_id"])] = row["abstract"]
    return lookup


def flatten_claims(claims: list[dict], corpus: dict[int, list[str]]) -> dict:
    premises, hypotheses, labels_out = [], [], []

    for claim in claims:
        claim_text = claim["claim"]
        evidence   = claim.get("evidence", {})

        if not evidence:
            nei_text = ""
            for did in claim.get("cited_doc_ids", []):
                abstract = corpus.get(int(did), [])
                if abstract:
                    nei_text = abstract[0]
                    break
            premises.append(claim_text)
            hypotheses.append(nei_text)
            labels_out.append(LABEL2ID["NOT_ENOUGH_INFO"])
            continue

        for doc_id_str, entries in evidence.items():
            doc_id   = int(doc_id_str)
            abstract = corpus.get(doc_id, [])

            for entry in entries:
                label_str = entry["label"]

                sent_indices  = entry["sentences"] if entry["sentences"] else [0]
                evidence_text = " ".join(
                    abstract[i] for i in sent_indices if i < len(abstract)
                ).strip()
                if not evidence_text:
                    evidence_text = abstract[0] if abstract else ""

                premises.append(claim_text)
                hypotheses.append(evidence_text)
                labels_out.append(LABEL2ID[label_str])

    return {"premise": premises, "hypothesis": hypotheses, "labels": labels_out}


def build_clean_splits(claims_train, claims_dev, corpus):
    dev_full   = flatten_claims(claims_dev,   corpus)
    train_full = flatten_claims(claims_train, corpus)

    dev_pair_set   = set(zip(dev_full["premise"],   dev_full["hypothesis"]))
    train_pair_set = set(zip(train_full["premise"], train_full["hypothesis"]))

    clean_train = {"premise": [], "hypothesis": [], "labels": []}
    train_dropped = 0
    for p, h, l in zip(train_full["premise"], train_full["hypothesis"], train_full["labels"]):
        if (p, h) in dev_pair_set:
            train_dropped += 1
        else:
            clean_train["premise"].append(p)
            clean_train["hypothesis"].append(h)
            clean_train["labels"].append(l)

    clean_dev = {"premise": [], "hypothesis": [], "labels": []}
    dev_dropped = 0
    for p, h, l in zip(dev_full["premise"], dev_full["hypothesis"], dev_full["labels"]):
        if (p, h) in train_pair_set:
            dev_dropped += 1
        else:
            clean_dev["premise"].append(p)
            clean_dev["hypothesis"].append(h)
            clean_dev["labels"].append(l)

    print(f"  Train rows after flatten         : {len(train_full['labels'])}")
    print(f"  Train rows dropped (in dev)      : {train_dropped}")
    print(f"  Train rows after dedup           : {len(clean_train['labels'])}")
    print(f"  Dev rows after flatten           : {len(dev_full['labels'])}")
    print(f"  Dev rows dropped (in train)      : {dev_dropped}")
    print(f"  Dev rows after dedup             : {len(clean_dev['labels'])}")

    t2d = sum(1 for p, h in zip(clean_train["premise"], clean_train["hypothesis"])
              if (p, h) in dev_pair_set)
    d2t = sum(1 for p, h in zip(clean_dev["premise"], clean_dev["hypothesis"])
              if (p, h) in train_pair_set)
    print(f"  Leak check train->dev (must be 0): {t2d}")
    print(f"  Leak check dev->train (must be 0): {d2t}")

    return clean_train, clean_dev

# ── Load files ────────────────────────────────────────────────────────────────
print("Loading corpus …")
corpus = build_corpus_lookup(SCIFACT_DIR / "corpus.jsonl")
print(f"  Corpus entries: {len(corpus)}")

print("Loading claim splits …")
claims_train = load_jsonl(SCIFACT_DIR / "claims_train.jsonl")
claims_dev   = load_jsonl(SCIFACT_DIR / "claims_dev.jsonl")
print(f"  Train claims: {len(claims_train)}  |  Dev claims: {len(claims_dev)}")

print("Building clean splits (removing doc_id overlap) …")
train_flat, dev_flat = build_clean_splits(claims_train, claims_dev, corpus)

print(f"  Train rows (after flatten): {len(train_flat['labels'])}")
print(f"  Dev rows   (after flatten): {len(dev_flat['labels'])}")

train_label_counts = Counter(train_flat["labels"])
print(f"\n  Train label distribution:")
for lid, name in ID2LABEL.items():
    print(f"    {name:20s} → {train_label_counts.get(lid, 0):4d} rows")

n_total   = len(train_flat["labels"])
n_classes = 3
weights   = [
    n_total / (n_classes * train_label_counts.get(i, 1))
    for i in range(n_classes)
]
class_weights = torch.tensor(weights, dtype=torch.float)
print(f"\n  Class weights: {[round(w, 3) for w in weights]}")

# ── Warmup steps ──────────────────────────────────────────────────────────────
TOTAL_STEPS  = (n_total // BATCH_SIZE) * EPOCHS
WARMUP_STEPS = int(WARMUP_RATIO * TOTAL_STEPS)
print(f"  Total train steps : {TOTAL_STEPS}")
print(f"  Warmup steps (20%): {WARMUP_STEPS}")

# ── Build HuggingFace Dataset objects ─────────────────────────────────────────
raw_train = Dataset.from_dict(train_flat)
raw_dev   = Dataset.from_dict(dev_flat)
print(f"\n  Dataset columns: {raw_train.column_names}")

# ══════════════════════════════════════════════════════════════════════════════
# TOKENISE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading tokenizer from {MULTINLI_CKPT} …")
tokenizer = AutoTokenizer.from_pretrained(MULTINLI_CKPT)

def tokenize(batch):
    enc = tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    enc["labels"] = batch["labels"]
    enc.pop("token_type_ids", None)
    return enc

print("Tokenising …")
dataset_train = raw_train.map(
    tokenize,
    batched=True,
    batch_size=256,
    remove_columns=raw_train.column_names,
)
dataset_dev = raw_dev.map(
    tokenize,
    batched=True,
    batch_size=256,
    remove_columns=raw_dev.column_names,
)

dataset_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
dataset_dev.set_format("torch",   columns=["input_ids", "attention_mask", "labels"])
print(f"  Columns after tokenise: {dataset_train.column_names}")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading model from {MULTINLI_CKPT} …")
model = AutoModelForSequenceClassification.from_pretrained(
    MULTINLI_CKPT,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print("  Model loaded. Classifier head re-labelled for SciFact schema.")

# ══════════════════════════════════════════════════════════════════════════════
# CUSTOM TRAINER
# ══════════════════════════════════════════════════════════════════════════════

class SciFActTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ══════════════════════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════════════════════

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds    = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    accuracy = (preds == labels).mean()
    return {"macro_f1": macro_f1, "accuracy": accuracy}

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING ARGS
# ══════════════════════════════════════════════════════════════════════════════

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=MAX_GRAD_NORM,

    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,

    logging_steps=50,
    log_level="error",
    log_level_replica="error",

    fp16=True,
    bf16=False,

    report_to="none",
)

# ══════════════════════════════════════════════════════════════════════════════
# TRAIN
# ══════════════════════════════════════════════════════════════════════════════

trainer = SciFActTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("\nStarting SciFact fine-tuning …")
print(f"  Epochs        : {EPOCHS}")
print(f"  LR            : {LR}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Class weights : {[round(w, 3) for w in weights]}")
print(f"  Expected time : ~25–35 min on Kaggle T4\n")

trainer.train()

# ══════════════════════════════════════════════════════════════════════════════
# SAVE BEST MODEL
# ══════════════════════════════════════════════════════════════════════════════

BEST_MODEL_DIR = "/kaggle/working/buss305-scifact-bestmodel"
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"\nBest model saved to {BEST_MODEL_DIR}")

# ══════════════════════════════════════════════════════════════════════════════
# FINAL EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Final evaluation (claims_dev.jsonl) ──")
preds_out = trainer.predict(dataset_dev)
preds_ids = np.argmax(preds_out.predictions, axis=-1)
true_ids  = preds_out.label_ids

print(classification_report(
    true_ids, preds_ids,
    target_names=["SUPPORT", "NOT_ENOUGH_INFO", "CONTRADICT"],
    digits=4,
))

macro_f1 = f1_score(true_ids, preds_ids, average="macro")
accuracy  = (preds_ids == true_ids).mean()
print(f"  eval_macro_f1 : {macro_f1:.4f}")
print(f"  eval_accuracy : {accuracy:.4f}")
print("\n✓ train_scifact.py complete.")

2026-06-11 17:50:27.972197: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781200228.156476      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781200228.212620      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781200228.652639      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781200228.652688      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781200228.652691      58 computation_placer.cc:177] computation placer alr

Loading corpus …
  Corpus entries: 5183
Loading claim splits …
  Train claims: 809  |  Dev claims: 300
Building clean splits (removing doc_id overlap) …
  Train rows after flatten         : 1261
  Train rows dropped (in dev)      : 2
  Train rows after dedup           : 1259
  Dev rows after flatten           : 450
  Dev rows dropped (in train)      : 2
  Dev rows after dedup             : 448
  Leak check train->dev (must be 0): 0
  Leak check dev->train (must be 0): 0
  Train rows (after flatten): 1259
  Dev rows   (after flatten): 448

  Train label distribution:
    SUPPORT              →  616 rows
    NOT_ENOUGH_INFO      →  302 rows
    CONTRADICT           →  341 rows

  Class weights: [0.681, 1.39, 1.231]
  Total train steps : 1256
  Warmup steps (20%): 125

  Dataset columns: ['premise', 'hypothesis', 'labels']

Loading tokenizer from /kaggle/input/models/minoola33/deberta-v3-base-multinli-nli/transformers/default/1 …
Tokenising …


Map:   0%|          | 0/1259 [00:00<?, ? examples/s]

Map:   0%|          | 0/448 [00:00<?, ? examples/s]

  Columns after tokenise: ['labels', 'input_ids', 'attention_mask']

Loading model from /kaggle/input/models/minoola33/deberta-v3-base-multinli-nli/transformers/default/1 …
  Model loaded. Classifier head re-labelled for SciFact schema.

Starting SciFact fine-tuning …
  Epochs        : 8
  LR            : 2e-06
  Batch size    : 8
  Class weights : [0.681, 1.39, 1.231]
  Expected time : ~25–35 min on Kaggle T4



Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.912900,0.803474,0.674066,0.671875
2,0.664000,0.603316,0.761155,0.763393
3,0.441200,0.527213,0.802844,0.808036
4,0.292100,0.501493,0.840475,0.845982
5,0.254200,0.503224,0.848762,0.854911
6,0.188400,0.492685,0.855253,0.861607
7,0.189900,0.500088,0.864050,0.870536
8,0.149700,0.492466,0.859487,0.866071



Best model saved to /kaggle/working/buss305-scifact-bestmodel

── Final evaluation (claims_dev.jsonl) ──


                 precision    recall  f1-score   support

        SUPPORT     0.8967    0.8843    0.8904       216
NOT_ENOUGH_INFO     0.8347    0.9182    0.8745       110
     CONTRADICT     0.8421    0.7869    0.8136       122

       accuracy                         0.8661       448
      macro avg     0.8578    0.8631    0.8595       448
   weighted avg     0.8666    0.8661    0.8656       448

  eval_macro_f1 : 0.8595
  eval_accuracy : 0.8661

✓ train_scifact.py complete.


#### Epoch 7 run will be saved.

In [10]:
# ── torch.load patch (suppress weights_only warning on older torch) ───────────
import torch
import copy

_orig_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# ── Standard imports ──────────────────────────────────────────────────────────
import json
import numpy as np
from pathlib import Path
from collections import Counter
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from sklearn.metrics import f1_score, classification_report
from torch.nn import CrossEntropyLoss

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# ── Paths ─────────────────────────────────────────────────────────────────────
MULTINLI_CKPT = "/kaggle/input/models/minoola33/deberta-v3-base-multinli-nli/transformers/default/1"
OUTPUT_DIR    = "/kaggle/working/scifact_checkpoint"

# Folder that contains claims_train.jsonl / claims_dev.jsonl / corpus.jsonl
SCIFACT_DIR   = Path("/kaggle/input/datasets/minoola33/scifact-data")

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LENGTH   = 256
BATCH_SIZE   = 8
EPOCHS       = 8
LR           = 2e-6
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

# ── Label mapping ─────────────────────────────────────────────────────────────
LABEL2ID = {"SUPPORT": 0, "NOT_ENOUGH_INFO": 1, "CONTRADICT": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_jsonl(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def build_corpus_lookup(corpus_path: Path) -> dict[int, list[str]]:
    lookup = {}
    for row in load_jsonl(corpus_path):
        lookup[int(row["doc_id"])] = row["abstract"]
    return lookup


def flatten_claims(claims: list[dict], corpus: dict[int, list[str]]) -> dict:
    premises, hypotheses, labels_out = [], [], []

    for claim in claims:
        claim_text = claim["claim"]
        evidence   = claim.get("evidence", {})

        if not evidence:
            nei_text = ""
            for did in claim.get("cited_doc_ids", []):
                abstract = corpus.get(int(did), [])
                if abstract:
                    nei_text = abstract[0]
                    break
            premises.append(claim_text)
            hypotheses.append(nei_text)
            labels_out.append(LABEL2ID["NOT_ENOUGH_INFO"])
            continue

        for doc_id_str, entries in evidence.items():
            doc_id   = int(doc_id_str)
            abstract = corpus.get(doc_id, [])

            for entry in entries:
                label_str = entry["label"]

                sent_indices  = entry["sentences"] if entry["sentences"] else [0]
                evidence_text = " ".join(
                    abstract[i] for i in sent_indices if i < len(abstract)
                ).strip()
                if not evidence_text:
                    evidence_text = abstract[0] if abstract else ""

                premises.append(claim_text)
                hypotheses.append(evidence_text)
                labels_out.append(LABEL2ID[label_str])

    return {"premise": premises, "hypothesis": hypotheses, "labels": labels_out}


def build_clean_splits(claims_train, claims_dev, corpus):
    dev_full   = flatten_claims(claims_dev,   corpus)
    train_full = flatten_claims(claims_train, corpus)

    dev_pair_set   = set(zip(dev_full["premise"],   dev_full["hypothesis"]))
    train_pair_set = set(zip(train_full["premise"], train_full["hypothesis"]))

    clean_train = {"premise": [], "hypothesis": [], "labels": []}
    train_dropped = 0
    for p, h, l in zip(train_full["premise"], train_full["hypothesis"], train_full["labels"]):
        if (p, h) in dev_pair_set:
            train_dropped += 1
        else:
            clean_train["premise"].append(p)
            clean_train["hypothesis"].append(h)
            clean_train["labels"].append(l)

    clean_dev = {"premise": [], "hypothesis": [], "labels": []}
    dev_dropped = 0
    for p, h, l in zip(dev_full["premise"], dev_full["hypothesis"], dev_full["labels"]):
        if (p, h) in train_pair_set:
            dev_dropped += 1
        else:
            clean_dev["premise"].append(p)
            clean_dev["hypothesis"].append(h)
            clean_dev["labels"].append(l)

    print(f"  Train rows after flatten         : {len(train_full['labels'])}")
    print(f"  Train rows dropped (in dev)      : {train_dropped}")
    print(f"  Train rows after dedup           : {len(clean_train['labels'])}")
    print(f"  Dev rows after flatten           : {len(dev_full['labels'])}")
    print(f"  Dev rows dropped (in train)      : {dev_dropped}")
    print(f"  Dev rows after dedup             : {len(clean_dev['labels'])}")

    t2d = sum(1 for p, h in zip(clean_train["premise"], clean_train["hypothesis"])
              if (p, h) in dev_pair_set)
    d2t = sum(1 for p, h in zip(clean_dev["premise"], clean_dev["hypothesis"])
              if (p, h) in train_pair_set)
    print(f"  Leak check train->dev (must be 0): {t2d}")
    print(f"  Leak check dev->train (must be 0): {d2t}")

    return clean_train, clean_dev

# ── Load files ────────────────────────────────────────────────────────────────
print("Loading corpus …")
corpus = build_corpus_lookup(SCIFACT_DIR / "corpus.jsonl")
print(f"  Corpus entries: {len(corpus)}")

print("Loading claim splits …")
claims_train = load_jsonl(SCIFACT_DIR / "claims_train.jsonl")
claims_dev   = load_jsonl(SCIFACT_DIR / "claims_dev.jsonl")
print(f"  Train claims: {len(claims_train)}  |  Dev claims: {len(claims_dev)}")

print("Building clean splits (removing doc_id overlap) …")
train_flat, dev_flat = build_clean_splits(claims_train, claims_dev, corpus)

print(f"  Train rows (after flatten): {len(train_flat['labels'])}")
print(f"  Dev rows   (after flatten): {len(dev_flat['labels'])}")

train_label_counts = Counter(train_flat["labels"])
print(f"\n  Train label distribution:")
for lid, name in ID2LABEL.items():
    print(f"    {name:20s} → {train_label_counts.get(lid, 0):4d} rows")

n_total   = len(train_flat["labels"])
n_classes = 3
weights   = [
    n_total / (n_classes * train_label_counts.get(i, 1))
    for i in range(n_classes)
]
class_weights = torch.tensor(weights, dtype=torch.float)
print(f"\n  Class weights: {[round(w, 3) for w in weights]}")

# ── Warmup steps ──────────────────────────────────────────────────────────────
TOTAL_STEPS  = (n_total // BATCH_SIZE) * EPOCHS
WARMUP_STEPS = int(WARMUP_RATIO * TOTAL_STEPS)
print(f"  Total train steps : {TOTAL_STEPS}")
print(f"  Warmup steps (20%): {WARMUP_STEPS}")

# ── Build HuggingFace Dataset objects ─────────────────────────────────────────
raw_train = Dataset.from_dict(train_flat)
raw_dev   = Dataset.from_dict(dev_flat)
print(f"\n  Dataset columns: {raw_train.column_names}")

# ══════════════════════════════════════════════════════════════════════════════
# TOKENISE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading tokenizer from {MULTINLI_CKPT} …")
tokenizer = AutoTokenizer.from_pretrained(MULTINLI_CKPT)

def tokenize(batch):
    enc = tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    enc["labels"] = batch["labels"]
    enc.pop("token_type_ids", None)
    return enc

print("Tokenising …")
dataset_train = raw_train.map(
    tokenize,
    batched=True,
    batch_size=256,
    remove_columns=raw_train.column_names,
)
dataset_dev = raw_dev.map(
    tokenize,
    batched=True,
    batch_size=256,
    remove_columns=raw_dev.column_names,
)

dataset_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
dataset_dev.set_format("torch",   columns=["input_ids", "attention_mask", "labels"])
print(f"  Columns after tokenise: {dataset_train.column_names}")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading model from {MULTINLI_CKPT} …")
model = AutoModelForSequenceClassification.from_pretrained(
    MULTINLI_CKPT,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print("  Model loaded. Classifier head re-labelled for SciFact schema.")

# ══════════════════════════════════════════════════════════════════════════════
# EPOCH 7 CAPTURE CALLBACK
# ══════════════════════════════════════════════════════════════════════════════

class SaveEpoch7Callback(TrainerCallback):
    def __init__(self):
        self.epoch7_state = None
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if round(state.epoch) == 7:
            self.epoch7_state = copy.deepcopy(trainer.model.state_dict())
            print(f"  ↑ Epoch 7 model captured (macro-F1: {metrics.get('eval_macro_f1', 0):.4f})")

epoch7_callback = SaveEpoch7Callback()

# ══════════════════════════════════════════════════════════════════════════════
# CUSTOM TRAINER
# ══════════════════════════════════════════════════════════════════════════════

class SciFActTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ══════════════════════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════════════════════

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds    = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    accuracy = (preds == labels).mean()
    return {"macro_f1": macro_f1, "accuracy": accuracy}

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING ARGS
# ══════════════════════════════════════════════════════════════════════════════

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=MAX_GRAD_NORM,

    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,

    logging_steps=50,
    log_level="error",
    log_level_replica="error",

    fp16=True,
    bf16=False,

    report_to="none",
)

# ══════════════════════════════════════════════════════════════════════════════
# TRAIN
# ══════════════════════════════════════════════════════════════════════════════

trainer = SciFActTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[epoch7_callback],
)

print("\nStarting SciFact fine-tuning …")
print(f"  Epochs        : {EPOCHS}")
print(f"  LR            : {LR}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Class weights : {[round(w, 3) for w in weights]}")
print(f"  Expected time : ~25–35 min on Kaggle T4\n")

trainer.train()

# ══════════════════════════════════════════════════════════════════════════════
# SAVE EPOCH 7 MODEL
# ══════════════════════════════════════════════════════════════════════════════

BEST_MODEL_DIR = "/kaggle/working/buss305-scifact-bestmodel"
trainer.model.load_state_dict(epoch7_callback.epoch7_state)
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"\nEpoch 7 model saved to {BEST_MODEL_DIR}")

# ══════════════════════════════════════════════════════════════════════════════
# FINAL EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Final evaluation (claims_dev.jsonl) ──")
preds_out = trainer.predict(dataset_dev)
preds_ids = np.argmax(preds_out.predictions, axis=-1)
true_ids  = preds_out.label_ids

print(classification_report(
    true_ids, preds_ids,
    target_names=["SUPPORT", "NOT_ENOUGH_INFO", "CONTRADICT"],
    digits=4,
))

macro_f1 = f1_score(true_ids, preds_ids, average="macro")
accuracy  = (preds_ids == true_ids).mean()
print(f"  eval_macro_f1 : {macro_f1:.4f}")
print(f"  eval_accuracy : {accuracy:.4f}")
print("\n✓ train_scifact.py complete.")

Loading corpus …
  Corpus entries: 5183
Loading claim splits …
  Train claims: 809  |  Dev claims: 300
Building clean splits (removing doc_id overlap) …
  Train rows after flatten         : 1261
  Train rows dropped (in dev)      : 2
  Train rows after dedup           : 1259
  Dev rows after flatten           : 450
  Dev rows dropped (in train)      : 2
  Dev rows after dedup             : 448
  Leak check train->dev (must be 0): 0
  Leak check dev->train (must be 0): 0
  Train rows (after flatten): 1259
  Dev rows   (after flatten): 448

  Train label distribution:
    SUPPORT              →  616 rows
    NOT_ENOUGH_INFO      →  302 rows
    CONTRADICT           →  341 rows

  Class weights: [0.681, 1.39, 1.231]
  Total train steps : 1256
  Warmup steps (20%): 125

  Dataset columns: ['premise', 'hypothesis', 'labels']

Loading tokenizer from /kaggle/input/models/minoola33/deberta-v3-base-multinli-nli/transformers/default/1 …
Tokenising …


Map:   0%|          | 0/1259 [00:00<?, ? examples/s]

Map:   0%|          | 0/448 [00:00<?, ? examples/s]

  Columns after tokenise: ['labels', 'input_ids', 'attention_mask']

Loading model from /kaggle/input/models/minoola33/deberta-v3-base-multinli-nli/transformers/default/1 …
  Model loaded. Classifier head re-labelled for SciFact schema.

Starting SciFact fine-tuning …
  Epochs        : 8
  LR            : 2e-06
  Batch size    : 8
  Class weights : [0.681, 1.39, 1.231]
  Expected time : ~25–35 min on Kaggle T4

{'loss': 2.4563, 'grad_norm': 24.433319091796875, 'learning_rate': 7.68e-07, 'epoch': 0.31645569620253167}
{'loss': 1.5086, 'grad_norm': 64.27272033691406, 'learning_rate': 1.552e-06, 'epoch': 0.6329113924050633}
{'loss': 0.9127, 'grad_norm': 11.915776252746582, 'learning_rate': 1.961369622475856e-06, 'epoch': 0.9493670886075949}
{'eval_loss': 0.8034214377403259, 'eval_macro_f1': 0.6740660340055357, 'eval_accuracy': 0.671875, 'eval_runtime': 3.6708, 'eval_samples_per_second': 122.044, 'eval_steps_per_second': 15.256, 'epoch': 1.0}
{'loss': 0.6975, 'grad_norm': 31.822288513183594

### Zero-Shot Evaluation on Scitail

In [13]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix
)

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

SCIFACT_MODEL = "/kaggle/working/buss305-scifact-bestmodel"
MAX_LENGTH    = 256
BATCH_SIZE    = 32

ID2LABEL = {0: "SUPPORT", 1: "NOT_ENOUGH_INFO", 2: "CONTRADICT"}

SCITAIL_LABEL_MAP = {
    "entailment": 0,
    "neutral":    1,
}

# ══════════════════════════════════════════════════════════════════════════════
# LOAD SCITAIL TEST SET
# ══════════════════════════════════════════════════════════════════════════════

print("Loading SciTail test set …")
scitail = load_dataset("allenai/scitail", "snli_format", split="test")
print(f"  SciTail test rows : {len(scitail)}")
print(f"  Columns           : {scitail.column_names}")
print(f"  Label values      : {set(scitail['gold_label'])}")

scitail = scitail.filter(lambda x: x["gold_label"] in SCITAIL_LABEL_MAP)
print(f"  Rows after filter : {len(scitail)}")

true_labels = [SCITAIL_LABEL_MAP[x] for x in scitail["gold_label"]]

from collections import Counter
dist = Counter(true_labels)
print(f"  Label distribution: SUPPORT={dist[0]}  NEI={dist[1]}")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD MODEL + TOKENIZER
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading model from {SCIFACT_MODEL} …")
tokenizer = AutoTokenizer.from_pretrained(SCIFACT_MODEL)
model     = AutoModelForSequenceClassification.from_pretrained(SCIFACT_MODEL)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"  Device: {device}")
print(f"  Model labels: {model.config.id2label}")

# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE
# ══════════════════════════════════════════════════════════════════════════════

print("\nRunning inference on SciTail test set …")

premises    = scitail["sentence1"]
hypotheses  = scitail["sentence2"]
all_preds   = []
all_logits  = []

with torch.no_grad():
    for i in range(0, len(premises), BATCH_SIZE):
        batch_p = premises[i : i + BATCH_SIZE]
        batch_h = hypotheses[i : i + BATCH_SIZE]

        enc = tokenizer(
            batch_p,
            batch_h,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        enc.pop("token_type_ids", None)
        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        logits  = outputs.logits.cpu().numpy()
        preds   = np.argmax(logits, axis=-1)

        all_preds.extend(preds.tolist())
        all_logits.extend(logits.tolist())

        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  {i}/{len(premises)} …")

print(f"  Done. {len(all_preds)} predictions made.")

# ══════════════════════════════════════════════════════════════════════════════
# RESULTS
# ══════════════════════════════════════════════════════════════════════════════

all_preds  = np.array(all_preds)
true_arr   = np.array(true_labels)

# ── 1. CONTRADICT rate ────────────────────────────────────────────────────────
contradict_count = (all_preds == 2).sum()
contradict_rate  = contradict_count / len(all_preds)
print(f"\n── CONTRADICT rate (should be low) ──────────────────────────")
print(f"  Model predicted CONTRADICT : {contradict_count} / {len(all_preds)} ({100*contradict_rate:.1f}%)")
if contradict_rate < 0.10:
    print(f"  ✓ Low CONTRADICT rate — model is not over-triggering contradiction")
elif contradict_rate < 0.20:
    print(f"  ⚠ Moderate CONTRADICT rate — model somewhat over-triggers contradiction")
else:
    print(f"  ✗ High CONTRADICT rate — model confused on binary-only domain")

# ── 2. Binary F1 ──────────────────────────────────────────────────────────────
binary_preds = np.where(all_preds == 0, 0, 1)
binary_true  = true_arr

binary_f1       = f1_score(binary_true, binary_preds, average="macro")
binary_accuracy = (binary_preds == binary_true).mean()

print(f"\n── Binary evaluation (SUPPORT vs NEI) ───────────────────────")
print(f"  Macro-F1  : {binary_f1:.4f}")
print(f"  Accuracy  : {binary_accuracy:.4f}")
print()
print(classification_report(
    binary_true, binary_preds,
    target_names=["SUPPORT (entailment)", "NEI (neutral)"],
    digits=4,
))

# ── 3. Full 3-class report ────────────────────────────────────────────────────
print(f"── Full 3-class prediction distribution ─────────────────────")
pred_dist = Counter(all_preds.tolist())
for lid in [0, 1, 2]:
    print(f"  {ID2LABEL[lid]:20s}: {pred_dist.get(lid, 0):5d}  ({100*pred_dist.get(lid,0)/len(all_preds):.1f}%)")

# ── 4. Confusion matrix ───────────────────────────────────────────────────────
print(f"\n── Confusion matrix (rows=true, cols=predicted) ─────────────")
print(f"  True labels: 0=SUPPORT(entailment)  1=NEI(neutral)")
print(f"  Pred labels: 0=SUPPORT  1=NEI  2=CONTRADICT")
cm = confusion_matrix(true_arr, all_preds, labels=[0, 1, 2])
print(f"             SUPPORT   NEI   CONTRADICT")
for i, row_name in enumerate(["SUPPORT(true)", "NEI(true)    "]):
    if i < len(cm):
        print(f"  {row_name}: {cm[i]}")

# ── 5. Interpretation ─────────────────────────────────────────────────────────
print(f"\n── Interpretation ───────────────────────────────────────────")
if binary_f1 >= 0.75:
    print(f"  ✓ STRONG transfer (binary F1={binary_f1:.3f})")
    print(f"    Model generalises scientific NLI reasoning beyond SciFact.")
elif binary_f1 >= 0.60:
    print(f"  ~ MODERATE transfer (binary F1={binary_f1:.3f})")
    print(f"    Model shows partial generalisation. Some SciFact-specific patterns.")
else:
    print(f"  ✗ WEAK transfer (binary F1={binary_f1:.3f})")
    print(f"    Model may have overfit SciFact domain.")

print("\n✓ eval_scitail.py complete.")

Loading SciTail test set …
  SciTail test rows : 2126
  Columns           : ['sentence1_binary_parse', 'sentence1_parse', 'sentence1', 'sentence2_parse', 'sentence2', 'annotator_labels', 'gold_label']
  Label values      : {'neutral', 'entailment'}
  Rows after filter : 2126
  Label distribution: SUPPORT=842  NEI=1284

Loading model from /kaggle/working/buss305-scifact-bestmodel …
  Device: cuda
  Model labels: {0: 'SUPPORT', 1: 'NOT_ENOUGH_INFO', 2: 'CONTRADICT'}

Running inference on SciTail test set …
  0/2126 …
  320/2126 …
  640/2126 …
  960/2126 …
  1280/2126 …
  1600/2126 …
  1920/2126 …
  Done. 2126 predictions made.

── CONTRADICT rate (should be low) ──────────────────────────
  Model predicted CONTRADICT : 174 / 2126 (8.2%)
  ✓ Low CONTRADICT rate — model is not over-triggering contradiction

── Binary evaluation (SUPPORT vs NEI) ───────────────────────
  Macro-F1  : 0.7713
  Accuracy  : 0.7714

                      precision    recall  f1-score   support

SUPPORT (entailme